## 하이퍼파라미터 튜닝: 최적의 모델을 찾는 탐색의 기술

### 개요

Day 2의 Part 1에서는 모델의 성능을 결정하는 '구조(Architecture)'에 대해 깊이 있게 배웠습니다. 

우리는 더 깊고 넓은 모델을 만들고, 가중치 초기화, 배치 정규화, 드롭아웃과 같은 기법들을 통해 모델을 정교하게 다듬었습니다.

하지만 정교하게 설계된 모델 구조도 올바른 "훈련 방법"이 있어야 그 잠재력을 모두 발휘할 수 있습니다. 

딥러닝 모델을 학습시킬 때, 우리는 학습 과정 자체를 제어하는 여러 '조절 손잡이' 즉, `하이퍼파라미터(Hyperparameter)`를 마주하게 됩니다. 

이 값들을 어떻게 설정하느냐에 따라 모델의 학습 속도, 안정성, 그리고 최종 성능이 극적으로 달라질 수 있습니다.

이번 파트에서는 모델의 성능을 한 단계 더 끌어올리기 위한 핵심 기술인 `하이퍼파라미터 튜닝`의 세계를 탐험합니다. 

"학습률은 얼마나 커야 할까?", "한 번에 몇 개의 데이터를 학습해야 할까?", "어떤 최적화 방법이 가장 효과적일까?"와 같은 질문에 대한 답을 찾아가는 여정이 될 것입니다.

`이번 파트의 학습 목표:`

  * 모델의 학습 과정을 제어하는 주요 하이퍼파라미터(학습률, 배치 크기, 옵티마이저 등)의 의미와 중요성을 이해합니다. 
  
  * 각 하이퍼파라미터 값의 변화가 모델 학습에 미치는 영향을 코드 실습을 통해 직접 확인하고 분석할 수 있습니다.
  * 과적합을 방지하고 최적의 학습 지점을 찾는 조기 종료(Early Stopping) 기법을 이해하고 적용할 수 있습니다.
  * 수동 탐색, 그리드 탐색과 같은 기본적인 하이퍼파라미터 튜닝 전략을 배우고 구현할 수 있습니다.
  * 이 모든 기법을 종합하여, 이전 파트에서 만든 모델의 성능을 체계적인 실험을 통해 개선하는 실전 경험을 쌓습니다.

이번 실습에서도 `위스콘신 유방암 데이터셋`을 계속 사용하여, 하이퍼파라미터 조정이 실제로 모델의 정확도를 얼마나 향상시키는지 직접 확인해 보겠습니다.

### 1. 하이퍼파라미터 vs. 파라미터: 무엇이 다른가?

본격적인 탐험에 앞서, 두 가지 용어를 명확히 구분해야 합니다.

  * `파라미터 (Parameter)`: 모델이 `학습을 통해 스스로 찾아내는 값`들입니다.  신경망의 가중치(weight)와 편향(bias)이 대표적입니다. 이 값들은 데이터로부터 모델 내부에서 결정됩니다.
  
  * `하이퍼파라미터 (Hyperparameter)`: 모델이 학습하기 `전에 사용자가 직접 설정해야 하는 값`들입니다.  학습률(learning rate), 배치 크기(batch size) 등이 여기에 속하며, 이 값들은 '어떻게' 학습할지를 결정합니다.

| 구분 | 모델 파라미터 (예: 가중치, 편향) | 하이퍼파라미터 (예: 학습률, 배치 크기) |
| :--- | :--- | :--- |
| `결정 주체` | 데이터 학습을 통해 `자동으로` 결정됩니다. | 학습 전에 `사용자가 직접` 설정합니다.  |
| `역할` | 모델 자체의 구성요소로 `예측에 사용`됩니다. | 모델 `훈련 과정을 제어`합니다.  |
| `예시` | 신경망 각 층의 가중치 값들 | 학습 속도를 조절하는 학습률 값, 한 번에 처리할 데이터 양인 배치 크기  |

우리의 목표는 최적의 '하이퍼파라미터' 조합을 찾아내어, 모델이 최상의 '파라미터'를 학습하도록 돕는 것입니다.


### 2. 가장 중요한 한 걸음: 학습률 (Learning Rate)

#### 2.1. 학습률의 의미와 중요성

하이퍼파라미터 중에서도 가장 중요하다고 알려진 것이 바로 `학습률(Learning Rate)`입니다.  

학습률은 모델이 손실 함수(loss function)의 최저점을 찾아가는 과정, 즉 경사 하강법(Gradient Descent)에서 `한 번에 얼마나 이동할지(step)를 결정하는 보폭의 크기`입니다. 

산 정상에서 안개를 뚫고 가장 낮은 계곡으로 내려가는 상황을 상상해 봅시다. 

  * `학습률이 너무 크면 (보폭이 너무 크면)`: 성큼성큼 걷다가 계곡의 바닥을 지나쳐 반대편으로 넘어가 버리거나, 계곡 주변을 왔다 갔다 하며 안절부절못할 수 있습니다.  이를 `발산(divergence)`이라고 하며, 손실(loss) 값이 줄어들지 않고 오히려 커지거나 요동치는 현상으로 나타납니다.
  
  * `학습률이 너무 작으면 (보폭이 너무 작으면)`: 너무 조심스럽게 종종걸음을 걷느라 계곡 바닥에 도달하기까지 매우 오랜 시간이 걸릴 수 있습니다.  또는, 진짜 계곡 바닥이 아닌 움푹 파인 작은 웅덩이(local minimum)에 빠져 만족하고 멈춰버릴 수도 있습니다.

따라서 너무 크지도, 작지도 않은 `'적절한' 학습률`을 찾는 것은 빠르고 안정적인 모델 학습의 핵심입니다. 

#### 2.2. 코드 실습: 학습률 변화에 따른 학습 곡선 관찰

Day 2-Part 1에서 만들었던 `AdvancedClassifier` 모델을 기반으로, 학습률 값에 따라 학습 곡선이 어떻게 변하는지 직접 눈으로 확인해 보겠습니다.

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import plotly.graph_objects as go

# 0. 데이터 준비 (Day2-Part1과 동일)
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

class BreastCancerDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)
    def __len__(self):
        return len(self.features)
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

train_dataset = BreastCancerDataset(X_train, y_train)
test_dataset = BreastCancerDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [9]:
# 1. Day2-Part1의 AdvancedClassifier 모델 정의
class AdvancedClassifier(nn.Module):
    def __init__(self, num_features, num_classes, dropout_p=0.4):
        super(AdvancedClassifier, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(num_features, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p=dropout_p),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(p=dropout_p),
            nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(p=dropout_p),
            nn.Linear(32, num_classes)
        )
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.kaiming_normal_(module.weight, mode='fan_in', nonlinearity='relu')
            if module.bias is not None:
                nn.init.constant_(module.bias, 0)

    def forward(self, x):
        return self.net(x)

In [10]:
# 2. 학습률 비교를 위한 학습 함수 정의
def train_for_lr_test(learning_rate):
    input_features = X_train.shape[1]
    model = AdvancedClassifier(num_features=input_features, num_classes=2)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    train_losses = []
    num_epochs = 40

    print(f"--- 학습 시작 (Learning Rate: {learning_rate}) ---")
    for epoch in range(num_epochs):
        model.train()
        epoch_train_loss = 0.0
        for features, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            epoch_train_loss += loss.item()

        avg_loss = epoch_train_loss / len(train_loader)
        train_losses.append(avg_loss)
        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}')

    return train_losses

In [11]:
# 3. 다양한 학습률로 모델 학습 및 결과 저장
lr_low_history = train_for_lr_test(learning_rate=1e-5) # 너무 낮은 학습률

--- 학습 시작 (Learning Rate: 1e-05) ---
Epoch [10/40], Loss: 0.9592
Epoch [20/40], Loss: 0.9214
Epoch [30/40], Loss: 0.9015
Epoch [40/40], Loss: 0.9396


In [12]:
lr_good_history = train_for_lr_test(learning_rate=1e-3) # 적절한 학습률

--- 학습 시작 (Learning Rate: 0.001) ---
Epoch [10/40], Loss: 0.2704
Epoch [20/40], Loss: 0.2765
Epoch [30/40], Loss: 0.1340
Epoch [40/40], Loss: 0.1295


In [13]:
lr_high_history = train_for_lr_test(learning_rate=1e-1) # 너무 높은 학습률

--- 학습 시작 (Learning Rate: 0.1) ---
Epoch [10/40], Loss: 0.1897
Epoch [20/40], Loss: 0.2082
Epoch [30/40], Loss: 0.4439
Epoch [40/40], Loss: 0.3201


In [14]:
# 4. 결과 시각화
fig = go.Figure()
epochs = list(range(1, 41))

fig.add_trace(go.Scatter(x=epochs, y=lr_low_history, name='Too Low (1e-5)', mode='lines+markers'))
fig.add_trace(go.Scatter(x=epochs, y=lr_good_history, name='Good (1e-3)', mode='lines+markers'))
fig.add_trace(go.Scatter(x=epochs, y=lr_high_history, name='Too High (1e-1)', mode='lines+markers'))

fig.update_layout(
    title='학습률(Learning Rate)에 따른 학습 손실(Loss) 변화',
    xaxis_title='에포크 (Epochs)',
    yaxis_title='학습 손실 (Training Loss)',
    yaxis_type="log", # 손실 값의 큰 차이를 보기 위해 y축을 로그 스케일로 설정
    legend_title="Learning Rate"
)
fig.show()

위 코드를 실행하면 세 가지 학습률에 대한 손실 그래프를 확인할 수 있습니다.

* `Too Low (1e-5)`: 손실이 거의 줄어들지 않고, 40 에포크가 지나도 여전히 높은 값을 유지하며 충분히 학습되지 않은 모습을 보입니다.

* `Good (1e-3)`: 손실이 초반에 빠르게 감소한 뒤, 이후에도 점진적으로 감소하며 비교적 안정적으로 수렴하는 모습을 보여줍니다.
* `Too High (1e-1)`: 손실 값이 불안정하게 크게 요동치며, 오히려 낮은 손실을 보일 때도 있지만 전반적으로 진동이 심하고 안정적으로 수렴하지 못하는 모습을 보입니다.

이처럼 시각화를 통해 학습률의 영향을 직접 확인하고, 최적의 값을 찾아가는 감을 익히는 것이 중요합니다.


### 3. 한 번에 얼마나 먹일까: 배치 크기 (Batch Size)

`배치 크기(Batch Size)`는 한 번의 가중치 업데이트, 즉 한 번의 경사 하강법 스텝에 사용되는 훈련 데이터 샘플의 개수를 의미합니다.  

전체 데이터셋을 작은 묶음(batch)으로 나누어 학습을 진행합니다.

시험공부에 비유해 볼까요? 

  * `배치 크기가 작을수록 (예: 1, 8, 16)`: 문제를 하나 풀고 바로 채점하고 오답노트를 작성하는 것과 같습니다. 
      
      * `장점`: 피드백이 잦아(가중치 업데이트가 빈번하여) 학습이 빠르게 진행되는 것처럼 보입니다. 또한, 각 배치의 노이즈가 모델이 local minima에 빠지는 것을 방지하고 더 좋은 해를 찾도록 돕는 `규제(regularization) 효과`를 주어 일반화 성능을 높일 수 있습니다. 
      
      * `단점`: 한 번의 업데이트가 전체 데이터의 경향을 제대로 반영하지 못해 학습이 불안정할 수 있습니다(손실 곡선이 심하게 요동침).
  
  * `배치 크기가 클수록 (예: 128, 256, 전체 데이터)`: 한 챕터를 모두 공부한 뒤 모의고사를 보고 한꺼번에 오답을 정리하는 것과 같습니다. 
      
      * `장점`: 전체 데이터의 경향을 더 잘 반영하므로 학습이 매우 안정적입니다. 병렬 연산에 유리하여 특정 하드웨어에서 학습 속도가 더 빠를 수 있습니다.
      
      * `단점`: 업데이트 빈도가 적어 수렴 속도가 느릴 수 있습니다.  또한, 너무 날카로운 최소점(sharp minima)으로 수렴하여 일반화 성능이 오히려 떨어질 수 있다는 연구 결과도 있습니다. GPU 메모리를 많이 차지하는 것도 단점입니다. 

대부분의 경우, 이 둘의 장점을 절충한 32, 64, 128 정도의 `미니배치(mini-batch)`를 사용하는 것이 일반적입니다. 

### 4. 언제까지 공부해야 할까: 에포크 (Epochs)와 조기 종료

`에포크(Epoch)`는 전체 훈련 데이터셋이 모델을 한 번 통과했음을 의미하는 단위입니다.  1000개의 훈련 데이터가 있고 배치 크기가 100이라면, 10개의 배치를 모두 처리했을 때 1 에포크가 완료됩니다.

에포크를 얼마나 많이 반복해야 할까요?

  * `너무 적은 에포크`: 모델이 데이터를 충분히 학습하지 못해 훈련 데이터조차 제대로 예측하지 못하는 `과소적합(Underfitting)` 상태가 됩니다. 
  
  * `너무 많은 에포크`: 모델이 훈련 데이터의 사소한 노이즈까지 모두 암기해버려, 새로운 데이터(검증/테스트 데이터)에 대한 성능은 오히려 떨어지는 `과적합(Overfitting)` 상태가 됩니다. 

그렇다면 최적의 에포크 횟수는 어떻게 알 수 있을까요? 무작정 큰 숫자를 설정하고 마냥 기다릴 수는 없습니다. 여기서 `조기 종료(Early Stopping)`라는 아주 실용적이고 강력한 기법이 등장합니다.

`조기 종료`는 훈련 과정에서 `검증 데이터(validation set)의 손실`을 계속 주시하다가, 이 손실이 더 이상 감소하지 않고 증가하기 시작하는 시점에 학습을 멈추는 기법입니다.  

이는 모델이 과적합을 시작하려는 바로 그 순간을 포착하여 학습을 중단시키는 효과적인 규제 방법입니다.


### 5. 어떤 길로 내려갈까: 옵티마이저 (Optimizer)

`옵티마이저(Optimizer)`는 계산된 경사(gradient)를 사용하여 모델의 가중치를 어떻게 업데이트할지를 결정하는 알고리즘입니다.  

어떤 옵티마이저를 사용하느냐에 따라 학습의 속도와 안정성이 크게 달라집니다.

자동차에 비유해 볼까요? 

  * `SGD (Stochastic Gradient Descent)`: 가장 기본적인 옵티마이저로, 수동 기어 자동차와 같습니다.  정해진 학습률(기어)로 꾸준히 나아가지만, 가파른 언덕이나 험난한 지형(복잡한 손실 함수 공간)을 만났을 때 유연하게 대처하기 어렵습니다.  Momentum, Nesterov Accelerated Gradient 등 변형들이 있지만, 여전히 하이퍼파라미터 설정에 민감한 편입니다. 
  
  * `Adam (Adaptive Moment Estimation)`: 현재 가장 널리 사용되는 옵티마이저 중 하나로, 자동 변속기와 최신 주행 보조 시스템을 갖춘 자동차와 같습니다. 
      
      * 각 파라미터마다 `개별적인 학습률을 적응적(Adaptive)으로 조절`해줍니다. 
      
      * 과거의 경사 방향을 참고하는 `모멘텀(Momentum)` 방식을 결합하여, 더 빠르고 안정적으로 최적점을 찾아갑니다. 

Adam은 초기 하이퍼파라미터 값에 비교적 덜 민감하고, 대부분의 문제에서 빠르고 안정적인 성능을 보여주기 때문에 `초심자에게 강력하게 추천되는 기본 옵티마이저`입니다. 


### 6. 최적의 조합을 찾는 전략: 하이퍼파라미터 튜닝 기법

지금까지 배운 여러 하이퍼파라미터들은 서로 상호작용하며 모델 성능에 영향을 줍니다. 

최적의 조합을 찾는 것은 매우 중요한 과제이며, 이를 위한 몇 가지 전략이 있습니다.

  * `수동 탐색 (Manual Tuning)`: 전문가의 직관과 경험에 의존하여 손으로 직접 값을 바꿔가며 실험하는 방법입니다.  학습 과정을 깊이 이해하는 데 도움이 되지만, 비체계적이고 시간이 많이 소요됩니다. 
  
  * `그리드 탐색 (Grid Search)`: 사용자가 지정한 하이퍼파라미터 값들의 모든 조합을 격자처럼 만들어 `전부 시도`하는 방법입니다.  예를 들어 학습률 `[0.1, 0.01]`, 배치 크기 `[32, 64]`를 테스트한다면 총 `2x2=4`개의 모든 조합을 실행합니다. 체계적이지만, 하이퍼파라미터의 종류나 후보 값이 늘어나면 시도해야 할 조합의 수가 기하급수적으로 증가하여 엄청난 계산 비용을 요구합니다. 
  * `랜덤 탐색 (Random Search)`: 그리드 탐색의 비효율성을 개선한 방법으로, 지정된 범위 내에서 하이퍼파라미터 조합을 `무작위로 샘플링`하여 시도하는 방법입니다.  연구에 따르면, 적은 횟수의 시도로도 그리드 탐색보다 더 좋은 조합을 찾을 확률이 높다고 알려져 있습니다.  왜냐하면 모든 하이퍼파라미터가 똑같이 중요하지 않기 때문에, 덜 중요한 파라미터에 자원을 낭비하는 대신 더 중요한 파라미터의 다양한 값을 탐색할 기회가 많아지기 때문입니다.
  * `베이지안 탐색 (Bayesian Optimization)`: 가장 정교한 하이퍼파라미터 튜닝 기법으로, 이전 시도들의 결과를 바탕으로 다음에 시도할 가장 유망한 조합을 지능적으로 예측하는 방법입니다. 확률적 모델(보통 가우시안 프로세스)을 사용하여 손실 함수의 형태를 추정하고, 획득 함수(Acquisition Function)를 통해 탐색과 활용의 균형을 맞춥니다. 계산 비용이 높지만, 적은 시도로도 최적의 조합을 찾을 가능성이 높아 고성능 컴퓨팅 환경에서 선호됩니다.

입문 단계에서는 `수동 탐색`으로 각 하이퍼파라미터의 영향을 먼저 파악하고, 이후 `그리드 탐색`이나 `랜덤 탐색`을 통해 체계적으로 최적화하는 것이 좋습니다. 

실무에서는 `Optuna`, `Hyperopt` 같은 라이브러리를 활용하여 자동화된 하이퍼파라미터 튜닝을 수행하기도 합니다.